# 06 — Full Model Training on Kaggle

This notebook performs **all trainable model fitting** for the article using a Kaggle GPU.

The proposed architecture is:

**SAMGT — Shamal-Aware Multi-Scale Graph Transformer**

The intended manuscript direction is:

**A Shamal-Transition-Aware Multi-Scale Graph Transformer for Multi-Horizon Wind Vector Forecasting over the Persian Gulf and Gulf of Oman**

This Kaggle version is designed so that:

- all required project data are read from `/kaggle/input/`,
- all trainable outputs are written to `/kaggle/working/`,
- the independent 2020 test set is **not evaluated here**,
- the best weight file for each model is saved automatically,
- training histories and exact experiment configuration are saved,
- all weights/logs are zipped at the end for download.

### Models trained

| Model | Role |
|---|---|
| `GRU` | conventional deep-learning baseline |
| `TransformerLSTM_2026` | recent Transformer–recurrent baseline |
| `GraphTransformer_2025` | recent graph + Transformer baseline |
| `MS_NoGraph` | multi-scale ablation |
| `MS_Graph_NoShamal` | graph ablation |
| `SAMGT_Full` | proposed Shamal-aware multi-scale graph Transformer |

For final article experiments keep `QUICK_MODE = False`.

The 2020 test period stays untouched until Notebook 07.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = Path(
    "/content/drive/MyDrive/kaggle_training_upload.zip"
)

EXTRACT_DIR = Path(
    "/content/drive/MyDrive/wind_training_data"
)

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

if EXTRACT_DIR.exists():
    print("Folder already exists:", EXTRACT_DIR)
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

shutil.unpack_archive(
    str(ZIP_PATH),
    str(EXTRACT_DIR)
)

print("Extraction finished.")
print("Extracted to:")
print(EXTRACT_DIR)

Extraction finished.
Extracted to:
/content/drive/MyDrive/wind_training_data


In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/wind_training_data/kaggle_training_upload"
)

## 1. Recent-model rationale

The architecture choices are motivated by recent literature rather than by adding arbitrary complexity.

Recent examples include:

- **Wu et al. (Applied Energy, 2025)** — Non-stationary GNNCrossformer, combining graph information with Transformer-based spatiotemporal forecasting. DOI: `10.1016/j.apenergy.2024.124492`.
- **Lu & Chen (Measurement, 2025)** — graph-based Transformer architecture for distributed wind-sequence reconstruction and forecasting. DOI: `10.1016/j.measurement.2025.117400`.
- **Luo et al. (Energy, 2025)** — dual-attention patch-based multi-scale Transformer for wind-speed forecasting. DOI: `10.1016/j.energy.2025.139178`.
- **Venkateswaran et al. (Journal of Wind Engineering and Industrial Aerodynamics, 2026)** — multi-horizon wind-speed forecasting using a CEEMDAN–Transformer–LSTM framework. DOI: `10.1016/j.jweia.2026.106408`.
- **Gomez et al. (Electric Power Systems Research, 2026)** — hybrid TCN, sLSTM, and multi-head self-attention for wind-speed prediction. DOI: `10.1016/j.epsr.2026.112945`.
- **Scientific Reports (2025)** also published Persian Gulf wind prediction using ERA5-derived wind components and a brain-emotional-learning-inspired model, showing that the region itself is publishable but also making it important that the present study contribute something beyond ordinary scalar wind-speed prediction.

The present implementation is an **adapted comparative framework**, not a claim that every baseline is a line-by-line reproduction of the cited papers. All models are trained on the same causal inputs, horizons, spatial domain, and chronological split so that the comparison remains controlled.

The proposed `SAMGT_Full` model is the article contribution to be tested. Its superiority is **not assumed**; it must be demonstrated in Notebook 07.


## Kaggle data you must upload

Before running this notebook, create **one Kaggle Dataset** containing the outputs from Notebooks 03–05 while preserving this project-relative folder structure:

```text
wind-paper-ready-data/
├── data/
│   ├── processed/
│   │   ├── study_grid.csv
│   │   ├── vector_features/
│   │   │   ├── 2005.nc
│   │   │   ├── 2006.nc
│   │   │   ├── ...
│   │   │   └── 2020.nc
│   │   └── shamal/
│   │       └── shamal_hourly_labels.nc
│   └── splits/
│       └── vector/
│           └── split_manifest.csv
└── outputs/
    ├── models/
    │   ├── vector_history_scaler.joblib
    │   ├── vector_target_scaler.joblib
    │   └── vector_spatial_scaler.joblib
    └── tables/
        ├── temporal_scale_schema.csv
        ├── forecast_horizon_schema.csv
        └── shamal_split_regime_summary.csv
```

### You do not need to upload for Notebook 06

You do **not** need the original raw NetCDF files, IHO shapefiles, old scalar feature files, project reports/PDFs, or Notebooks 01–05 themselves.

### Why include 2020?

Notebook 06 trains on 2005–2017 and validates on 2018–2019. It does not evaluate 2020. The 2020 feature file is included so the same Kaggle Dataset can later be attached to Notebook 07 for the untouched final evaluation.

### Kaggle setup

1. Create the Kaggle Dataset with the folder tree above.
2. Open this notebook in Kaggle.
3. Use **Add Input** and attach that dataset.
4. Enable a GPU accelerator.
5. Run all cells.
6. Weights are written to `/kaggle/working/wind_training_outputs/weights/`.
7. The final cell creates downloadable weight and ZIP files.


## 2. Imports, GPU configuration, and reproducibility

A GPU is strongly recommended.

The training notebook treats each forecast issue time as one sample containing the complete 512-cell marine field. This is preferable to flattening the data into independent grid-cell samples because the graph models need simultaneous spatial information.

Mixed-precision training is enabled automatically when CUDA is available.


In [2]:
from pathlib import Path
import gc
import json
import math
import random
import time
from copy import deepcopy

import joblib
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive


# ---------------------------------------------------------
# Mount Google Drive
# ---------------------------------------------------------

drive.mount(
    "/content/drive"
)


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/wind_training_data/kaggle_training_upload"
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/wind_model_results"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


assert PROJECT_ROOT.exists(), (
    f"Project folder not found:\n{PROJECT_ROOT}"
)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Output root:",
    OUTPUT_ROOT
)


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ---------------------------------------------------------
# Device
# ---------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print(
    "\nPyTorch version:",
    torch.__version__
)

print(
    "Device:",
    DEVICE
)


if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA capability:",
        torch.cuda.get_device_capability(0)
    )

    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        )
    )

else:

    print(
        "\nWARNING: No CUDA GPU detected."
    )

    print(
        "In Colab go to:"
    )

    print(
        "Runtime -> Change runtime type -> GPU"
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/wind_training_data/kaggle_training_upload
Output root: /content/drive/MyDrive/wind_model_results

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
CUDA capability: (7, 5)
GPU memory (GB): 14.56


## 3. Project paths and training configuration

The notebook consumes only outputs already created by Notebooks 03–05.

Important training defaults:

- maximum 20 epochs,
- early stopping,
- AdamW optimizer,
- mixed precision on GPU,
- gradient clipping,
- validation-based learning-rate reduction,
- model checkpointing,
- no test-set access for model selection.

`QUICK_MODE` exists only for debugging the pipeline. It must be `False` for the article experiments.


In [ ]:
from pathlib import Path
import zipfile
import shutil


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

ZIP_PATH = Path(
    "/content/drive/MyDrive/kaggle_training_upload.zip"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/wind_training_data/kaggle_training_upload"
)

VECTOR_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vector_features"
)

VECTOR_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Check ZIP
# ---------------------------------------------------------

assert ZIP_PATH.exists(), (
    f"ZIP file not found:\n{ZIP_PATH}"
)

print("ZIP found:")
print(ZIP_PATH)


# ---------------------------------------------------------
# Recover missing annual files directly from ZIP
# ---------------------------------------------------------

years_to_recover = [
    2014,
    2020,
]

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    zip_files = zf.namelist()

    for year in years_to_recover:

        target_name = (
            f"data/processed/vector_features/{year}.nc"
        )

        matches = [
            name
            for name in zip_files
            if name.endswith(target_name)
        ]

        if len(matches) == 0:

            print(
                f"\nERROR: {year}.nc is NOT inside the ZIP."
            )

            continue

        source_in_zip = matches[0]

        destination = (
            VECTOR_FEATURE_DIR
            / f"{year}.nc"
        )

        print(
            f"\nRecovering {year}.nc..."
        )

        print(
            "ZIP location:",
            source_in_zip
        )

        with zf.open(
            source_in_zip
        ) as source:

            with open(
                destination,
                "wb"
            ) as output:

                shutil.copyfileobj(
                    source,
                    output
                )

        print(
            "Saved to:",
            destination
        )

        print(
            "Size:",
            round(
                destination.stat().st_size
                / (1024 ** 2),
                2
            ),
            "MB"
        )


# ---------------------------------------------------------
# Verify all vector feature years again
# ---------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "VERIFYING VECTOR FEATURE FILES"
)

print(
    "=" * 70
)


expected_years = list(
    range(
        2005,
        2021
    )
)

missing_years = []


for year in expected_years:

    file_path = (
        VECTOR_FEATURE_DIR
        / f"{year}.nc"
    )

    if file_path.exists():

        size_mb = (
            file_path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{year}: OK  ({size_mb:.2f} MB)"
        )

    else:

        print(
            f"{year}: MISSING"
        )

        missing_years.append(
            year
        )


print(
    "\n"
    + "=" * 70
)


if len(missing_years) == 0:

    print(
        "SUCCESS: All 2005-2020 vector feature files are present."
    )

else:

    print(
        "Still missing:",
        missing_years
    )

    print(
        "\nThose files are also missing from the uploaded ZIP."
    )

    print(
        "We will need to upload only those missing .nc files "
        "from your computer."
    )

ZIP found:
/content/drive/MyDrive/kaggle_training_upload.zip

ERROR: 2014.nc is NOT inside the ZIP.

ERROR: 2020.nc is NOT inside the ZIP.

VERIFYING VECTOR FEATURE FILES
2005: OK  (304.82 MB)
2006: OK  (311.75 MB)
2007: OK  (311.50 MB)
2008: OK  (311.13 MB)
2009: OK  (311.01 MB)
2010: OK  (312.28 MB)
2011: OK  (310.53 MB)
2012: OK  (313.84 MB)
2013: OK  (310.45 MB)
2014: OK  (311.05 MB)
2015: OK  (311.89 MB)
2016: OK  (311.66 MB)
2017: OK  (312.29 MB)
2018: OK  (312.46 MB)
2019: OK  (312.32 MB)
2020: OK  (68.75 MB)

SUCCESS: All 2005-2020 vector feature files are present.


In [21]:
# ---------------------------------------------------------
# Google Colab + Google Drive project paths
# ---------------------------------------------------------

from pathlib import Path


# The extracted training package in Google Drive.
#
# Expected structure:
#
# MyDrive/
# └── wind_training_data/
#     └── kaggle_training_upload/
#         ├── data/
#         └── outputs/
#
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/wind_training_data/kaggle_training_upload"
)

# Keep this alias because later cells in the original notebook
# may refer to INPUT_PROJECT_ROOT.
INPUT_PROJECT_ROOT = PROJECT_ROOT


# ---------------------------------------------------------
# Verify project root
# ---------------------------------------------------------

assert PROJECT_ROOT.exists(), (
    "Project root was not found.\n"
    f"Expected location:\n{PROJECT_ROOT}\n\n"
    "Make sure Google Drive is mounted and the ZIP was "
    "extracted to MyDrive/wind_training_data/."
)


# ---------------------------------------------------------
# Input data paths
# ---------------------------------------------------------

VECTOR_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vector_features"
)

STUDY_GRID_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "study_grid.csv"
)

SPLIT_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "vector"
    / "split_manifest.csv"
)

TEMPORAL_SCALE_SCHEMA_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "temporal_scale_schema.csv"
)

FORECAST_HORIZON_SCHEMA_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "forecast_horizon_schema.csv"
)

SHAMAL_LABELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "shamal"
    / "shamal_hourly_labels.nc"
)

SHAMAL_SPLIT_SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "shamal_split_regime_summary.csv"
)

HISTORY_SCALER_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "models"
    / "vector_history_scaler.joblib"
)

TARGET_SCALER_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "models"
    / "vector_target_scaler.joblib"
)

SPATIAL_SCALER_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "models"
    / "vector_spatial_scaler.joblib"
)


# ---------------------------------------------------------
# Persistent Google Drive output paths
# ---------------------------------------------------------

# All training results will be saved permanently in Google Drive.
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/wind_model_results"
)

CHECKPOINT_DIR = (
    OUTPUT_ROOT
    / "weights"
)

TRAINING_TABLE_DIR = (
    OUTPUT_ROOT
    / "training_tables"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAINING_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Verify required input files/directories
# ---------------------------------------------------------

required_paths = {
    "Vector feature directory":
        VECTOR_FEATURE_DIR,

    "Study grid":
        STUDY_GRID_PATH,

    "Split manifest":
        SPLIT_MANIFEST_PATH,

    "Temporal scale schema":
        TEMPORAL_SCALE_SCHEMA_PATH,

    "Forecast horizon schema":
        FORECAST_HORIZON_SCHEMA_PATH,

    "Shamal labels":
        SHAMAL_LABELS_PATH,

    "Shamal split summary":
        SHAMAL_SPLIT_SUMMARY_PATH,

    "History scaler":
        HISTORY_SCALER_PATH,

    "Target scaler":
        TARGET_SCALER_PATH,

    "Spatial scaler":
        SPATIAL_SCALER_PATH,
}


missing_paths = []

for name, path in required_paths.items():

    if not path.exists():

        missing_paths.append(
            (name, path)
        )


if missing_paths:

    print(
        "\nERROR: Some required files are missing:\n"
    )

    for name, path in missing_paths:

        print(
            f"{name}:\n"
            f"  {path}\n"
        )

    raise FileNotFoundError(
        "Required training files are missing. "
        "Check the extracted Google Drive folder."
    )

else:

    print(
        "All required project files were found."
    )


# ---------------------------------------------------------
# Verify annual vector feature files
# ---------------------------------------------------------

expected_years = list(
    range(
        2005,
        2021
    )
)

missing_years = []

for year in expected_years:

    year_file = (
        VECTOR_FEATURE_DIR
        / f"{year}.nc"
    )

    if not year_file.exists():

        missing_years.append(
            year
        )


if missing_years:

    raise FileNotFoundError(
        "Missing annual vector feature files for years: "
        + ", ".join(
            map(
                str,
                missing_years
            )
        )
    )


print(
    "Vector feature files:",
    f"{expected_years[0]}-{expected_years[-1]}",
    "OK"
)


# ---------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------

QUICK_MODE = False

# Number of forecast issue-times per batch.
#
# If Colab GPU runs out of memory:
# first try 8, then 4.
BATCH_HOURS = 16

MAX_EPOCHS = 20

EARLY_STOPPING_PATIENCE = 4

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

GRAD_CLIP_NORM = 1.0

LR_REDUCTION_FACTOR = 0.5

LR_REDUCTION_PATIENCE = 2

MIN_LEARNING_RATE = 1e-6

HUBER_BETA = 0.5

REGIME_LOSS_WEIGHT = 0.15

CLASS_WEIGHT_POWER = 0.5

D_MODEL = 64

N_HEADS = 4

N_TEMPORAL_LAYERS = 2

GRAPH_LAYERS = 2

DROPOUT = 0.10

K_NEIGHBORS = 8

QUICK_HOURS_PER_YEAR = 256


# ---------------------------------------------------------
# Models to train
# ---------------------------------------------------------

MODELS_TO_TRAIN = [
    #"GRU",
    #"TransformerLSTM_2026",
    #"GraphTransformer_2025",
    "MS_NoGraph",
    "MS_Graph_NoShamal",
    "SAMGT_Full",
]


# ---------------------------------------------------------
# Configuration summary
# ---------------------------------------------------------

print(
    "\nProject root:"
)

print(
    PROJECT_ROOT
)


print(
    "\nVector feature directory:"
)

print(
    VECTOR_FEATURE_DIR
)


print(
    "\nPersistent output root:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nCheckpoint directory:"
)

print(
    CHECKPOINT_DIR
)


print(
    "\nTraining table directory:"
)

print(
    TRAINING_TABLE_DIR
)


print(
    "\nTraining configuration:"
)

print(
    "Quick mode:",
    QUICK_MODE
)

print(
    "Models to train:",
    MODELS_TO_TRAIN
)

print(
    "Batch issue-times:",
    BATCH_HOURS
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Early-stopping patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "d_model:",
    D_MODEL
)

print(
    "Attention heads:",
    N_HEADS
)

print(
    "Temporal layers:",
    N_TEMPORAL_LAYERS
)

print(
    "Graph layers:",
    GRAPH_LAYERS
)

print(
    "Graph neighbors:",
    K_NEIGHBORS
)

print(
    "Mixed precision:",
    AMP_ENABLED
)

All required project files were found.
Vector feature files: 2005-2020 OK

Project root:
/content/drive/MyDrive/wind_training_data/kaggle_training_upload

Vector feature directory:
/content/drive/MyDrive/wind_training_data/kaggle_training_upload/data/processed/vector_features

Persistent output root:
/content/drive/MyDrive/wind_model_results

Checkpoint directory:
/content/drive/MyDrive/wind_model_results/weights

Training table directory:
/content/drive/MyDrive/wind_model_results/training_tables

Training configuration:
Quick mode: False
Models to train: ['MS_NoGraph', 'MS_Graph_NoShamal', 'SAMGT_Full']
Batch issue-times: 16
Maximum epochs: 20
Early-stopping patience: 4
Learning rate: 0.0003
d_model: 64
Attention heads: 4
Temporal layers: 2
Graph layers: 2
Graph neighbors: 8
Mixed precision: True


## Kaggle input audit

This cell checks every file needed for training before any GPU time is spent.

It expects all vector feature files from 2005–2020 plus the study grid, split manifest, feature schemas, Shamal labels, Shamal split summary, and the three scalers.

Notebook 06 only loads 2005–2019 feature arrays during optimization. 2020 remains reserved for Notebook 07.


In [4]:
required_files = [
    STUDY_GRID_PATH,
    SPLIT_MANIFEST_PATH,
    TEMPORAL_SCALE_SCHEMA_PATH,
    FORECAST_HORIZON_SCHEMA_PATH,
    SHAMAL_LABELS_PATH,
    SHAMAL_SPLIT_SUMMARY_PATH,
    HISTORY_SCALER_PATH,
    TARGET_SCALER_PATH,
    SPATIAL_SCALER_PATH,
]

for year in range(2005, 2021):
    required_files.append(
        VECTOR_FEATURE_DIR / f"{year}.nc"
    )

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    print("Missing files:")
    for path in missing_files:
        print(" -", path)
    raise FileNotFoundError(
        "Kaggle input dataset is incomplete. "
        "Upload the missing files before training."
    )

print("Kaggle input audit: PASSED")
print("Required files found:", len(required_files))

print("\\nVector feature files:")
for year in range(2005, 2021):
    path = VECTOR_FEATURE_DIR / f"{year}.nc"
    size_gb = path.stat().st_size / (1024 ** 3)
    print(f"{year}: {size_gb:.3f} GB")

print("\\n2020 is present for Notebook 07 but is not used for training/model selection here.")


Kaggle input audit: PASSED
Required files found: 25
\nVector feature files:
2005: 0.298 GB
2006: 0.304 GB
2007: 0.304 GB
2008: 0.304 GB
2009: 0.304 GB
2010: 0.305 GB
2011: 0.303 GB
2012: 0.306 GB
2013: 0.303 GB
2014: 0.304 GB
2015: 0.305 GB
2016: 0.304 GB
2017: 0.305 GB
2018: 0.305 GB
2019: 0.305 GB
2020: 0.067 GB
\n2020 is present for Notebook 07 but is not used for training/model selection here.


## 4. Load experiment metadata, scalers, and Shamal targets

The exact lag groups and forecast horizons are read from the files produced by Notebook 03.

The split manifest comes from Notebook 04.

Shamal ground-truth targets come from Notebook 05.

The 2020 split is loaded only as metadata here so that the notebook can verify its existence. No 2020 feature arrays or 2020 metrics are used during training or validation.


In [5]:
required_paths = [
    STUDY_GRID_PATH,
    SPLIT_MANIFEST_PATH,
    TEMPORAL_SCALE_SCHEMA_PATH,
    FORECAST_HORIZON_SCHEMA_PATH,
    SHAMAL_LABELS_PATH,
    SHAMAL_SPLIT_SUMMARY_PATH,
    HISTORY_SCALER_PATH,
    TARGET_SCALER_PATH,
    SPATIAL_SCALER_PATH,
]

for path in required_paths:
    assert path.exists(), (
        f"Missing required input: {path}"
    )


study_grid = pd.read_csv(
    STUDY_GRID_PATH
).sort_values(
    "study_id"
).reset_index(
    drop=True
)

split_manifest = pd.read_csv(
    SPLIT_MANIFEST_PATH,
    parse_dates=[
        "issue_start",
        "issue_end",
    ]
)

temporal_scale_schema = pd.read_csv(
    TEMPORAL_SCALE_SCHEMA_PATH
)

forecast_horizon_schema = pd.read_csv(
    FORECAST_HORIZON_SCHEMA_PATH
)

shamal_split_summary = pd.read_csv(
    SHAMAL_SPLIT_SUMMARY_PATH
)


history_scaler = joblib.load(
    HISTORY_SCALER_PATH
)

target_scaler = joblib.load(
    TARGET_SCALER_PATH
)

spatial_scaler = joblib.load(
    SPATIAL_SCALER_PATH
)


with xr.open_dataset(
    SHAMAL_LABELS_PATH
) as ds_regime:
    shamal_labels = ds_regime.load()


ALL_LAGS = sorted(
    temporal_scale_schema[
        "lag_hours"
    ]
    .astype(int)
    .unique()
    .tolist()
)

FORECAST_HORIZONS = sorted(
    forecast_horizon_schema[
        "horizon_hours"
    ]
    .astype(int)
    .unique()
    .tolist()
)


SCALE_TO_LAGS = {
    scale_name: (
        temporal_scale_schema.loc[
            temporal_scale_schema[
                "temporal_scale"
            ] == scale_name,
            "lag_hours"
        ]
        .astype(int)
        .tolist()
    )
    for scale_name in [
        "short",
        "diurnal",
        "synoptic",
    ]
}


LAG_TO_INDEX = {
    lag: index
    for index, lag in enumerate(
        ALL_LAGS
    )
}

SCALE_TO_INDICES = {
    scale_name: [
        LAG_TO_INDEX[lag]
        for lag in scale_lags
    ]
    for scale_name, scale_lags in SCALE_TO_LAGS.items()
}


TRAIN_ROWS = (
    split_manifest[
        split_manifest["split"] == "train"
    ]
    .sort_values("year")
    .reset_index(drop=True)
)

VALIDATION_ROWS = (
    split_manifest[
        split_manifest["split"] == "validation"
    ]
    .sort_values("year")
    .reset_index(drop=True)
)

TEST_ROWS = (
    split_manifest[
        split_manifest["split"] == "test"
    ]
    .sort_values("year")
    .reset_index(drop=True)
)


assert TRAIN_ROWS["year"].tolist() == list(
    range(2005, 2018)
)

assert VALIDATION_ROWS["year"].tolist() == [
    2018,
    2019,
]

assert TEST_ROWS["year"].tolist() == [
    2020,
]


N_NODES = len(study_grid)
N_LAGS = len(ALL_LAGS)
N_HORIZONS = len(FORECAST_HORIZONS)


print("Nodes:", N_NODES)
print("Lags:", ALL_LAGS)
print("Horizons:", FORECAST_HORIZONS)

print("\nScale groups:")
for name, values in SCALE_TO_LAGS.items():
    print(name, "->", values)

print(
    "\nTraining years:",
    TRAIN_ROWS["year"].tolist()
)

print(
    "Validation years:",
    VALIDATION_ROWS["year"].tolist()
)

print(
    "Reserved test year:",
    TEST_ROWS["year"].tolist()
)

print(
    "\n2020 remains reserved for Notebook 07."
)


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Nodes: 512
Lags: [0, 1, 2, 3, 6, 12, 24, 48, 72, 120, 168]
Horizons: [1, 3, 6, 12, 24]

Scale groups:
short -> [0, 1, 2, 3, 6]
diurnal -> [12, 24, 48]
synoptic -> [72, 120, 168]

Training years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
Validation years: [2018, 2019]
Reserved test year: [2020]

2020 remains reserved for Notebook 07.


## 5. Static spatial features and K-nearest-neighbor marine graph

The 512 study cells are treated as spatial nodes.

A fixed K-nearest-neighbor graph is built from latitude/longitude using great-circle distance. The graph is used only to define which neighboring marine cells can exchange information; all graph attention weights are learned by the neural network.

Static node features contain:

- standardized latitude,
- standardized longitude,
- basin indicator (`0 = Persian Gulf`, `1 = Gulf of Oman`).

The graph is deterministic and is saved for reproducibility.


In [6]:
# ---------------------------------------------------------
# Static node features
# ---------------------------------------------------------

lat = study_grid[
    "latitude"
].to_numpy(
    dtype=np.float64
)

lon = study_grid[
    "longitude"
].to_numpy(
    dtype=np.float64
)

spatial_scaled = spatial_scaler.transform(
    np.column_stack([
        lat,
        lon,
    ])
).astype(
    np.float32
)

basin_code = (
    study_grid[
        "basin"
    ]
    .map({
        "Persian Gulf": 0.0,
        "Gulf of Oman": 1.0,
    })
    .to_numpy(
        dtype=np.float32
    )
    .reshape(-1, 1)
)

STATIC_NODE_FEATURES = np.concatenate(
    [
        spatial_scaled,
        basin_code,
    ],
    axis=1
).astype(
    np.float32
)

STATIC_DIM = (
    STATIC_NODE_FEATURES.shape[1]
)


# ---------------------------------------------------------
# Persian Gulf mask for the global Shamal head
# ---------------------------------------------------------

PERSIAN_GULF_MASK = (
    study_grid["basin"]
    .eq("Persian Gulf")
    .to_numpy()
)

assert PERSIAN_GULF_MASK.sum() == 357


# ---------------------------------------------------------
# Great-circle KNN graph
# ---------------------------------------------------------

coordinates_rad = np.radians(
    np.column_stack([
        lat,
        lon,
    ])
)

knn = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,
    metric="haversine",
)

knn.fit(
    coordinates_rad
)

distances_rad, neighbor_indices = (
    knn.kneighbors(
        coordinates_rad
    )
)

# Remove self-neighbor.
neighbor_indices = (
    neighbor_indices[:, 1:]
    .astype(np.int64)
)

distances_rad = (
    distances_rad[:, 1:]
)

EARTH_RADIUS_KM = 6371.0088

distances_km = (
    distances_rad
    * EARTH_RADIUS_KM
)


assert neighbor_indices.shape == (
    N_NODES,
    K_NEIGHBORS
)


# ---------------------------------------------------------
# Save graph audit table
# ---------------------------------------------------------

graph_rows = []

for node_index in range(N_NODES):
    for rank in range(K_NEIGHBORS):
        neighbor_index = int(
            neighbor_indices[
                node_index,
                rank
            ]
        )

        graph_rows.append({
            "study_id": int(
                study_grid.loc[
                    node_index,
                    "study_id"
                ]
            ),
            "neighbor_rank": rank + 1,
            "neighbor_study_id": int(
                study_grid.loc[
                    neighbor_index,
                    "study_id"
                ]
            ),
            "distance_km": float(
                distances_km[
                    node_index,
                    rank
                ]
            ),
        })


graph_table = pd.DataFrame(
    graph_rows
)

GRAPH_TABLE_PATH = (
    TRAINING_TABLE_DIR
    / "knn_graph_edges.csv"
)

GRAPH_INDEX_PATH = (
    CHECKPOINT_DIR
    / "knn_neighbor_indices.npy"
)

graph_table.to_csv(
    GRAPH_TABLE_PATH,
    index=False
)

np.save(
    GRAPH_INDEX_PATH,
    neighbor_indices
)


print(
    "Static feature shape:",
    STATIC_NODE_FEATURES.shape
)

print(
    "Neighbor-index shape:",
    neighbor_indices.shape
)

print(
    "Median neighbor distance (km):",
    round(
        float(
            np.median(
                distances_km
            )
        ),
        2
    )
)

print(
    "Maximum KNN-edge distance (km):",
    round(
        float(
            np.max(
                distances_km
            )
        ),
        2
    )
)

print(
    "\nSaved graph table:",
    GRAPH_TABLE_PATH.resolve()
)


Static feature shape: (512, 3)
Neighbor-index shape: (512, 8)
Median neighbor distance (km): 37.03
Maximum KNN-edge distance (km): 93.95

Saved graph table: /content/drive/MyDrive/wind_model_results/training_tables/knn_graph_edges.csv


## 6. Training-only Shamal class weights

The Shamal auxiliary classification task is imbalanced.

Class weights are calculated **only from training-period future regime targets**. Validation and test labels do not influence these weights.

A square-root inverse-frequency weighting is used rather than full inverse-frequency weighting so rare transition classes receive additional importance without allowing a very rare class to dominate optimization.


In [7]:
training_regime_counts = (
    shamal_split_summary[
        (
            shamal_split_summary["split"]
            == "train"
        )
        &
        (
            shamal_split_summary["target_type"]
            == "future_regime_target"
        )
    ]
    .groupby(
        "regime_code"
    )[
        "class_count"
    ]
    .sum()
    .reindex(
        [0, 1, 2, 3],
        fill_value=0
    )
)


if (
    training_regime_counts
    <= 0
).any():
    raise RuntimeError(
        "At least one Shamal regime class has zero "
        "training examples. Do not train the Shamal-aware "
        "model until Notebook 05's event definition and "
        "event counts have been scientifically reviewed."
    )


frequencies = (
    training_regime_counts.to_numpy(
        dtype=np.float64
    )
    / training_regime_counts.sum()
)

class_weights = (
    1.0
    / np.power(
        frequencies,
        CLASS_WEIGHT_POWER
    )
)

class_weights = (
    class_weights
    / class_weights.mean()
)

REGIME_CLASS_WEIGHTS = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


class_weight_table = pd.DataFrame({
    "regime_code": [0, 1, 2, 3],
    "training_count": training_regime_counts.values,
    "training_fraction": frequencies,
    "loss_weight": class_weights,
})


CLASS_WEIGHT_PATH = (
    TRAINING_TABLE_DIR
    / "shamal_training_class_weights.csv"
)

class_weight_table.to_csv(
    CLASS_WEIGHT_PATH,
    index=False
)


print("Training-only regime class weights:")
display(class_weight_table)

print("Saved:", CLASS_WEIGHT_PATH.resolve())


Training-only regime class weights:


,regime_code,training_count,training_fraction,loss_weight
0,0,523800,0.920886,0.212757
1,1,13860,0.024367,1.307934
2,2,17280,0.030380,1.171374
3,3,13860,0.024367,1.307934


Saved: /content/drive/MyDrive/wind_model_results/training_tables/shamal_training_class_weights.csv


## 7. Efficient year-wise batch loader

The model sees the complete 512-cell field for each forecast issue time.

For one batch:

- historical wind tensor: `[batch, 512, 11, 2]`,
- future vector target: `[batch, 512, 5, 2]`,
- calendar tensor: `[batch, 4]`,
- Shamal regime target: `[batch, 5]`.

The loader opens one annual file at a time, loads the selected split interval into memory, applies the training-derived scalers, shuffles training issue times, and yields GPU-ready batches.

This avoids creating another large copy of the scaled NetCDF dataset.


In [8]:
HISTORY_MEAN = np.asarray(
    history_scaler.mean_,
    dtype=np.float32
)

HISTORY_SCALE = np.asarray(
    history_scaler.scale_,
    dtype=np.float32
)

TARGET_MEAN = np.asarray(
    target_scaler.mean_,
    dtype=np.float32
)

TARGET_SCALE = np.asarray(
    target_scaler.scale_,
    dtype=np.float32
)


STATIC_NODE_TENSOR = torch.tensor(
    STATIC_NODE_FEATURES,
    dtype=torch.float32,
    device=DEVICE
)

NEIGHBOR_INDEX_TENSOR = torch.tensor(
    neighbor_indices,
    dtype=torch.long,
    device=DEVICE
)

PG_MASK_TENSOR = torch.tensor(
    PERSIAN_GULF_MASK,
    dtype=torch.bool,
    device=DEVICE
)


def load_year_split_arrays(row):
    """
    Load one annual split interval and return scaled model arrays.
    """

    feature_path = (
        INPUT_PROJECT_ROOT
        / row.feature_file
    )

    issue_start = pd.Timestamp(
        row.issue_start
    )

    issue_end = pd.Timestamp(
        row.issue_end
    )

    with xr.open_dataset(
        feature_path
    ) as ds:

        subset = (
            ds
            .sel(
                valid_time=slice(
                    issue_start,
                    issue_end
                )
            )
            .load()
        )

    valid_time = pd.DatetimeIndex(
        pd.to_datetime(
            subset[
                "valid_time"
            ].values
        )
    )

    u_history = (
        subset[
            "u10_history"
        ]
        .values
        .astype(
            np.float32
        )
    )

    v_history = (
        subset[
            "v10_history"
        ]
        .values
        .astype(
            np.float32
        )
    )

    history = np.stack(
        [
            u_history,
            v_history,
        ],
        axis=-1
    )

    history[
        ...,
        0
    ] = (
        history[
            ...,
            0
        ]
        - HISTORY_MEAN[0]
    ) / HISTORY_SCALE[0]

    history[
        ...,
        1
    ] = (
        history[
            ...,
            1
        ]
        - HISTORY_MEAN[1]
    ) / HISTORY_SCALE[1]


    u_target = (
        subset[
            "u10_target"
        ]
        .values
        .astype(
            np.float32
        )
    )

    v_target = (
        subset[
            "v10_target"
        ]
        .values
        .astype(
            np.float32
        )
    )

    target = np.stack(
        [
            u_target,
            v_target,
        ],
        axis=-1
    )

    target[
        ...,
        0
    ] = (
        target[
            ...,
            0
        ]
        - TARGET_MEAN[0]
    ) / TARGET_SCALE[0]

    target[
        ...,
        1
    ] = (
        target[
            ...,
            1
        ]
        - TARGET_MEAN[1]
    ) / TARGET_SCALE[1]


    calendar = np.column_stack([
        subset[
            "hour_sin"
        ].values,
        subset[
            "hour_cos"
        ].values,
        subset[
            "doy_sin"
        ].values,
        subset[
            "doy_cos"
        ].values,
    ]).astype(
        np.float32
    )


    regime_target = (
        shamal_labels[
            "regime_target"
        ]
        .sel(
            valid_time=valid_time
        )
        .values
        .astype(
            np.int64
        )
    )


    assert history.shape == (
        len(valid_time),
        N_NODES,
        N_LAGS,
        2
    )

    assert target.shape == (
        len(valid_time),
        N_NODES,
        N_HORIZONS,
        2
    )

    assert calendar.shape == (
        len(valid_time),
        4
    )

    assert regime_target.shape == (
        len(valid_time),
        N_HORIZONS
    )

    assert (
        regime_target
        >= 0
    ).all()

    return {
        "time": valid_time,
        "history": history,
        "target": target,
        "calendar": calendar,
        "regime_target": regime_target,
    }


def iter_split_batches(
    rows,
    batch_size,
    shuffle,
    epoch_seed,
):
    """
    Stream one split year-by-year.
    """

    rng = np.random.default_rng(
        epoch_seed
    )

    row_indices = np.arange(
        len(rows)
    )

    if shuffle:
        rng.shuffle(
            row_indices
        )

    for row_index in row_indices:

        row = rows.iloc[
            row_index
        ]

        arrays = load_year_split_arrays(
            row
        )

        n_times = len(
            arrays["time"]
        )

        indices = np.arange(
            n_times
        )

        if QUICK_MODE:
            indices = indices[
                :min(
                    QUICK_HOURS_PER_YEAR,
                    n_times
                )
            ]

        if shuffle:
            rng.shuffle(
                indices
            )

        for start in range(
            0,
            len(indices),
            batch_size
        ):

            batch_indices = indices[
                start:
                start + batch_size
            ]

            history = torch.from_numpy(
                arrays[
                    "history"
                ][
                    batch_indices
                ]
            ).to(
                DEVICE,
                non_blocking=True
            )

            target = torch.from_numpy(
                arrays[
                    "target"
                ][
                    batch_indices
                ]
            ).to(
                DEVICE,
                non_blocking=True
            )

            calendar = torch.from_numpy(
                arrays[
                    "calendar"
                ][
                    batch_indices
                ]
            ).to(
                DEVICE,
                non_blocking=True
            )

            regime_target = torch.from_numpy(
                arrays[
                    "regime_target"
                ][
                    batch_indices
                ]
            ).to(
                DEVICE,
                non_blocking=True
            )

            yield {
                "history": history,
                "target": target,
                "calendar": calendar,
                "regime_target": regime_target,
            }

        del arrays
        gc.collect()


## 8. Shared neural-network components

The models below share common building blocks so that differences between models are easier to interpret.

The proposed model uses:

1. scale-specific temporal attention,
2. learned fusion of short/diurnal/synoptic representations,
3. KNN graph attention,
4. residual feed-forward blocks,
5. auxiliary Shamal-regime prediction,
6. predicted-regime-conditioned multi-horizon wind-vector decoding.


In [9]:
class ResidualFeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        dropout,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(
            d_model
        )

        self.ff = nn.Sequential(
            nn.Linear(
                d_model,
                2 * d_model
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                2 * d_model,
                d_model
            ),
            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        x
    ):
        return x + self.ff(
            self.norm(
                x
            )
        )


class TemporalEncoder(nn.Module):

    def __init__(
        self,
        n_steps,
        lag_values,
        d_model,
        n_heads,
        n_layers,
        dropout,
    ):
        super().__init__()

        self.n_steps = n_steps

        lag_values = np.asarray(
            lag_values,
            dtype=np.float32
        )

        lag_norm = (
            np.log1p(
                lag_values
            )
            / np.log1p(
                max(
                    1.0,
                    float(
                        max(
                            ALL_LAGS
                        )
                    )
                )
            )
        )

        self.register_buffer(
            "lag_norm",
            torch.tensor(
                lag_norm,
                dtype=torch.float32
            ).view(
                1,
                1,
                n_steps,
                1
            )
        )

        self.input_projection = nn.Linear(
            3,
            d_model
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                n_steps,
                d_model
            )
        )

        nn.init.normal_(
            self.position_embedding,
            std=0.02
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=(
                    4 * d_model
                ),
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=n_layers,
            )
        )

        self.output_norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        x
    ):
        # x:
        # [B, N, S, 2]

        B, N, S, _ = x.shape

        lag_feature = (
            self.lag_norm
            .expand(
                B,
                N,
                S,
                1
            )
        )

        token_input = torch.cat(
            [
                x,
                lag_feature
            ],
            dim=-1
        )

        token = self.input_projection(
            token_input
        )

        token = token.reshape(
            B * N,
            S,
            -1
        )

        token = (
            token
            + self.position_embedding
        )

        token = self.encoder(
            token
        )

        token = self.output_norm(
            token
        )

        pooled = token.mean(
            dim=1
        )

        return pooled.reshape(
            B,
            N,
            -1
        )


class MultiScaleTemporalEncoder(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_layers,
        dropout,
    ):
        super().__init__()

        self.scale_names = [
            "short",
            "diurnal",
            "synoptic",
        ]

        self.scale_indices = {
            name: SCALE_TO_INDICES[
                name
            ]
            for name in self.scale_names
        }

        self.encoders = nn.ModuleDict({
            name: TemporalEncoder(
                n_steps=len(
                    self.scale_indices[
                        name
                    ]
                ),
                lag_values=SCALE_TO_LAGS[
                    name
                ],
                d_model=d_model,
                n_heads=n_heads,
                n_layers=n_layers,
                dropout=dropout,
            )
            for name in self.scale_names
        })

        self.scale_gate = nn.Sequential(
            nn.Linear(
                d_model,
                d_model // 2
            ),
            nn.GELU(),
            nn.Linear(
                d_model // 2,
                1
            ),
        )

        self.residual = (
            ResidualFeedForward(
                d_model=d_model,
                dropout=dropout,
            )
        )

    def forward(
        self,
        history
    ):
        scale_features = []

        for name in self.scale_names:

            index = self.scale_indices[
                name
            ]

            x_scale = history[
                :,
                :,
                index,
                :
            ]

            scale_features.append(
                self.encoders[
                    name
                ](
                    x_scale
                )
            )

        stacked = torch.stack(
            scale_features,
            dim=2
        )

        gate_logits = self.scale_gate(
            stacked
        ).squeeze(
            -1
        )

        gate_weights = torch.softmax(
            gate_logits,
            dim=2
        )

        fused = (
            stacked
            * gate_weights.unsqueeze(
                -1
            )
        ).sum(
            dim=2
        )

        fused = self.residual(
            fused
        )

        return (
            fused,
            gate_weights
        )


class KNNGraphAttentionBlock(nn.Module):

    def __init__(
        self,
        d_model,
        neighbor_index,
        dropout,
    ):
        super().__init__()

        self.register_buffer(
            "neighbor_index",
            neighbor_index.clone()
        )

        self.query = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.key = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.value = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.out = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.ff = (
            ResidualFeedForward(
                d_model=d_model,
                dropout=dropout,
            )
        )

        self.scale = math.sqrt(
            d_model
        )

    def forward(
        self,
        x
    ):
        # x:
        # [B, N, D]

        neighbors = x[
            :,
            self.neighbor_index,
            :
        ]

        q = self.query(
            x
        ).unsqueeze(
            2
        )

        k = self.key(
            neighbors
        )

        v = self.value(
            neighbors
        )

        attention_logits = (
            q
            * k
        ).sum(
            dim=-1
        ) / self.scale

        attention = torch.softmax(
            attention_logits,
            dim=-1
        )

        message = (
            attention.unsqueeze(
                -1
            )
            * v
        ).sum(
            dim=2
        )

        x = self.norm1(
            x
            + self.dropout(
                self.out(
                    message
                )
            )
        )

        x = self.ff(
            x
        )

        return x


class StaticContext(nn.Module):

    def __init__(
        self,
        static_dim,
        d_model,
    ):
        super().__init__()

        self.spatial_projection = nn.Sequential(
            nn.Linear(
                static_dim,
                d_model
            ),
            nn.GELU(),
            nn.Linear(
                d_model,
                d_model
            ),
        )

        self.calendar_projection = nn.Sequential(
            nn.Linear(
                4,
                d_model
            ),
            nn.GELU(),
            nn.Linear(
                d_model,
                d_model
            ),
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        node_feature,
        static_node,
        calendar,
    ):
        static_embedding = (
            self.spatial_projection(
                static_node
            )
            .unsqueeze(
                0
            )
        )

        calendar_embedding = (
            self.calendar_projection(
                calendar
            )
            .unsqueeze(
                1
            )
        )

        return self.norm(
            node_feature
            + static_embedding
            + calendar_embedding
        )


## 9. Baseline and proposed architectures

All trainable models predict the same five future wind vectors:

\[
(\hat u,\hat v)_{t+h},
\qquad
h=\{1,3,6,12,24\}\ \mathrm{h}.
\]

The ablation sequence is intentionally nested:

\[
\text{MS\_NoGraph}
\rightarrow
\text{MS\_Graph\_NoShamal}
\rightarrow
\text{SAMGT\_Full}.
\]

This lets the paper quantify the contribution of:

1. spatial graph interaction,
2. Shamal-transition awareness,

without changing the remainder of the architecture.


In [10]:
class GRUVectorForecaster(nn.Module):

    def __init__(
        self,
        d_model,
        dropout,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=3,
            hidden_size=d_model,
            num_layers=2,
            dropout=dropout,
            batch_first=True,
        )

        lag_values = np.asarray(
            ALL_LAGS[::-1],
            dtype=np.float32
        )

        lag_norm = (
            np.log1p(
                lag_values
            )
            / np.log1p(
                float(
                    max(
                        ALL_LAGS
                    )
                )
            )
        )

        self.register_buffer(
            "lag_norm",
            torch.tensor(
                lag_norm,
                dtype=torch.float32
            ).view(
                1,
                1,
                N_LAGS,
                1
            )
        )

        self.context = StaticContext(
            STATIC_DIM,
            d_model
        )

        self.head = nn.Sequential(
            nn.Linear(
                d_model,
                d_model
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                d_model,
                N_HORIZONS * 2
            ),
        )

    def forward(
        self,
        history,
        static_node,
        calendar,
    ):
        # Reverse lag order so sequence runs oldest -> current.
        x = history.flip(
            dims=[2]
        )

        B, N, L, _ = x.shape

        lag = self.lag_norm.expand(
            B,
            N,
            L,
            1
        )

        x = torch.cat(
            [
                x,
                lag
            ],
            dim=-1
        )

        x = x.reshape(
            B * N,
            L,
            3
        )

        _, hidden = self.gru(
            x
        )

        node = hidden[
            -1
        ].reshape(
            B,
            N,
            -1
        )

        node = self.context(
            node,
            static_node,
            calendar
        )

        output = self.head(
            node
        )

        return {
            "vector": output.view(
                B,
                N,
                N_HORIZONS,
                2
            ),
            "regime_logits": None,
            "scale_weights": None,
        }


class TransformerLSTMForecaster(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            3,
            d_model
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                N_LAGS,
                d_model
            )
        )

        nn.init.normal_(
            self.position_embedding,
            std=0.02
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=(
                    4 * d_model
                ),
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=2,
            )
        )

        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=1,
            batch_first=True,
        )

        self.context = StaticContext(
            STATIC_DIM,
            d_model
        )

        self.head = nn.Sequential(
            ResidualFeedForward(
                d_model,
                dropout
            ),
            nn.Linear(
                d_model,
                N_HORIZONS * 2
            ),
        )

        chronological_lags = np.asarray(
            ALL_LAGS[::-1],
            dtype=np.float32
        )

        lag_norm = (
            np.log1p(
                chronological_lags
            )
            / np.log1p(
                float(
                    max(
                        ALL_LAGS
                    )
                )
            )
        )

        self.register_buffer(
            "lag_norm",
            torch.tensor(
                lag_norm,
                dtype=torch.float32
            ).view(
                1,
                1,
                N_LAGS,
                1
            )
        )

    def forward(
        self,
        history,
        static_node,
        calendar,
    ):
        x = history.flip(
            dims=[2]
        )

        B, N, L, _ = x.shape

        lag = self.lag_norm.expand(
            B,
            N,
            L,
            1
        )

        x = torch.cat(
            [
                x,
                lag
            ],
            dim=-1
        )

        x = self.input_projection(
            x
        )

        x = x.reshape(
            B * N,
            L,
            -1
        )

        x = (
            x
            + self.position_embedding
        )

        x = self.transformer(
            x
        )

        _, (
            h,
            _
        ) = self.lstm(
            x
        )

        node = h[
            -1
        ].reshape(
            B,
            N,
            -1
        )

        node = self.context(
            node,
            static_node,
            calendar
        )

        output = self.head(
            node
        )

        return {
            "vector": output.view(
                B,
                N,
                N_HORIZONS,
                2
            ),
            "regime_logits": None,
            "scale_weights": None,
        }


class GraphTransformerForecaster(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_layers,
        graph_layers,
        dropout,
        neighbor_index,
    ):
        super().__init__()

        self.temporal = TemporalEncoder(
            n_steps=N_LAGS,
            lag_values=ALL_LAGS,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
        )

        self.context = StaticContext(
            STATIC_DIM,
            d_model
        )

        self.graph_blocks = nn.ModuleList([
            KNNGraphAttentionBlock(
                d_model=d_model,
                neighbor_index=neighbor_index,
                dropout=dropout,
            )
            for _ in range(
                graph_layers
            )
        ])

        self.head = nn.Sequential(
            ResidualFeedForward(
                d_model,
                dropout
            ),
            nn.Linear(
                d_model,
                N_HORIZONS * 2
            ),
        )

    def forward(
        self,
        history,
        static_node,
        calendar,
    ):
        B, N, _, _ = history.shape

        node = self.temporal(
            history
        )

        node = self.context(
            node,
            static_node,
            calendar
        )

        for block in self.graph_blocks:
            node = block(
                node
            )

        output = self.head(
            node
        )

        return {
            "vector": output.view(
                B,
                N,
                N_HORIZONS,
                2
            ),
            "regime_logits": None,
            "scale_weights": None,
        }


class SAMGT(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_layers,
        graph_layers,
        dropout,
        neighbor_index,
        pg_mask,
        use_graph,
        use_shamal,
    ):
        super().__init__()

        self.use_graph = use_graph
        self.use_shamal = use_shamal

        self.multiscale = (
            MultiScaleTemporalEncoder(
                d_model=d_model,
                n_heads=n_heads,
                n_layers=n_layers,
                dropout=dropout,
            )
        )

        self.context = StaticContext(
            STATIC_DIM,
            d_model
        )

        if use_graph:
            self.graph_blocks = nn.ModuleList([
                KNNGraphAttentionBlock(
                    d_model=d_model,
                    neighbor_index=neighbor_index,
                    dropout=dropout,
                )
                for _ in range(
                    graph_layers
                )
            ])
        else:
            self.graph_blocks = nn.ModuleList()

        self.register_buffer(
            "pg_mask",
            pg_mask.clone()
        )

        self.horizon_embedding = nn.Parameter(
            torch.zeros(
                N_HORIZONS,
                d_model
            )
        )

        nn.init.normal_(
            self.horizon_embedding,
            std=0.02
        )

        if use_shamal:
            self.regime_head = nn.Sequential(
                ResidualFeedForward(
                    d_model,
                    dropout
                ),
                nn.Linear(
                    d_model,
                    N_HORIZONS * 4
                ),
            )

            self.regime_projection = nn.Sequential(
                nn.Linear(
                    4,
                    d_model
                ),
                nn.GELU(),
                nn.Linear(
                    d_model,
                    d_model
                ),
            )
        else:
            self.regime_head = None
            self.regime_projection = None

        self.decoder = nn.Sequential(
            nn.Linear(
                d_model,
                d_model
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            ResidualFeedForward(
                d_model,
                dropout
            ),
            nn.Linear(
                d_model,
                2
            ),
        )

    def forward(
        self,
        history,
        static_node,
        calendar,
    ):
        B, N, _, _ = history.shape

        node, scale_weights = (
            self.multiscale(
                history
            )
        )

        node = self.context(
            node,
            static_node,
            calendar
        )

        for block in self.graph_blocks:
            node = block(
                node
            )

        horizon_context = (
            self.horizon_embedding
            .view(
                1,
                1,
                N_HORIZONS,
                -1
            )
        )

        decoder_feature = (
            node.unsqueeze(
                2
            )
            + horizon_context
        )

        regime_logits = None

        if self.use_shamal:

            regional_feature = (
                node[
                    :,
                    self.pg_mask,
                    :
                ]
                .mean(
                    dim=1
                )
            )

            regime_logits = (
                self.regime_head(
                    regional_feature
                )
                .view(
                    B,
                    N_HORIZONS,
                    4
                )
            )

            # Important:
            # decoder uses PREDICTED probabilities,
            # never true future regime labels.
            regime_probability = (
                torch.softmax(
                    regime_logits,
                    dim=-1
                )
            )

            regime_context = (
                self.regime_projection(
                    regime_probability
                )
                .unsqueeze(
                    1
                )
            )

            decoder_feature = (
                decoder_feature
                + regime_context
            )

        vector = self.decoder(
            decoder_feature
        )

        return {
            "vector": vector,
            "regime_logits": regime_logits,
            "scale_weights": scale_weights,
        }


## 10. Model registry and article ablation design

The registry trains all models under the same split, targets, scalers, batch definition, optimizer family, and early-stopping rule.

The three multi-scale entries provide the main ablation chain:

- `MS_NoGraph`
- `MS_Graph_NoShamal`
- `SAMGT_Full`

Only the final model has access to the **predicted** Shamal regime probabilities. True future Shamal labels are used only in the auxiliary classification loss.


In [18]:
def build_model(
    model_name
):

    # if model_name == "GRU":
    #     return GRUVectorForecaster(
    #         d_model=D_MODEL,
    #         dropout=DROPOUT,
    #     )

    # if model_name == "TransformerLSTM_2026":
    #     return TransformerLSTMForecaster(
    #         d_model=D_MODEL,
    #         n_heads=N_HEADS,
    #         dropout=DROPOUT,
    #     )

    # if model_name == "GraphTransformer_2025":
    #     return GraphTransformerForecaster(
    #         d_model=D_MODEL,
    #         n_heads=N_HEADS,
    #         n_layers=N_TEMPORAL_LAYERS,
    #         graph_layers=GRAPH_LAYERS,
    #         dropout=DROPOUT,
    #         neighbor_index=NEIGHBOR_INDEX_TENSOR,
    #     )

    if model_name == "MS_NoGraph":
        return SAMGT(
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_TEMPORAL_LAYERS,
            graph_layers=GRAPH_LAYERS,
            dropout=DROPOUT,
            neighbor_index=NEIGHBOR_INDEX_TENSOR,
            pg_mask=PG_MASK_TENSOR,
            use_graph=False,
            use_shamal=False,
        )

    if model_name == "MS_Graph_NoShamal":
        return SAMGT(
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_TEMPORAL_LAYERS,
            graph_layers=GRAPH_LAYERS,
            dropout=DROPOUT,
            neighbor_index=NEIGHBOR_INDEX_TENSOR,
            pg_mask=PG_MASK_TENSOR,
            use_graph=True,
            use_shamal=False,
        )

    if model_name == "SAMGT_Full":
        return SAMGT(
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_TEMPORAL_LAYERS,
            graph_layers=GRAPH_LAYERS,
            dropout=DROPOUT,
            neighbor_index=NEIGHBOR_INDEX_TENSOR,
            pg_mask=PG_MASK_TENSOR,
            use_graph=True,
            use_shamal=True,
        )

    raise ValueError(
        f"Unknown model: {model_name}"
    )


MODEL_NAMES = [
#    "GRU",
#    "TransformerLSTM_2026",
#    "GraphTransformer_2025",
    "MS_NoGraph",
    "MS_Graph_NoShamal",
    "SAMGT_Full",
]


model_registry_rows = []

for model_name in MODEL_NAMES:

    model = build_model(
        model_name
    )

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_count = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    model_registry_rows.append({
        "model": model_name,
        "parameters": parameter_count,
        "trainable_parameters": trainable_count,
        "uses_multiscale": model_name.startswith("MS_")
            or model_name == "SAMGT_Full",
        "uses_graph": model_name in [
            "GraphTransformer_2025",
            "MS_Graph_NoShamal",
            "SAMGT_Full",
        ],
        "uses_shamal_auxiliary_task": (
            model_name == "SAMGT_Full"
        ),
    })

    del model


model_registry = pd.DataFrame(
    model_registry_rows
)


MODEL_REGISTRY_PATH = (
    TRAINING_TABLE_DIR
    / "model_registry.csv"
)

model_registry.to_csv(
    MODEL_REGISTRY_PATH,
    index=False
)


display(model_registry)

print(
    "Saved:",
    MODEL_REGISTRY_PATH.resolve()
)


/tmp/ipykernel_1721/824970097.py:125: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


,model,parameters,trainable_parameters,uses_multiscale,uses_graph,uses_shamal_auxiliary_task
0,MS_NoGraph,350915,350915,True,False,False
1,MS_Graph_NoShamal,417475,417475,True,True,False
2,SAMGT_Full,439959,439959,True,True,True


Saved: /content/drive/MyDrive/wind_model_results/training_tables/model_registry.csv


## 11. Loss functions

The primary forecasting objective is Huber loss on standardized future \(u_{10}\) and \(v_{10}\) components.

For `SAMGT_Full`, the total training loss is:

\[
\mathcal{L}
=
\mathcal{L}_{vector}
+
\lambda
\mathcal{L}_{regime}.
\]

The regime term is weighted multi-class cross entropy using training-only class weights.

The model checkpoint criterion remains **validation vector loss**, not classification loss, because accurate wind-vector forecasting is the primary scientific task.


In [12]:
def compute_losses(
    output,
    target_vector,
    target_regime,
):
    vector_loss = F.smooth_l1_loss(
        output[
            "vector"
        ],
        target_vector,
        beta=HUBER_BETA,
    )

    regime_logits = output[
        "regime_logits"
    ]

    if regime_logits is None:

        regime_loss = torch.tensor(
            0.0,
            device=target_vector.device
        )

        total_loss = vector_loss

    else:

        regime_loss = F.cross_entropy(
            regime_logits.reshape(
                -1,
                4
            ),
            target_regime.reshape(
                -1
            ),
            weight=REGIME_CLASS_WEIGHTS,
        )

        total_loss = (
            vector_loss
            + REGIME_LOSS_WEIGHT
            * regime_loss
        )

    return {
        "total": total_loss,
        "vector": vector_loss,
        "regime": regime_loss,
    }


## 12. Validation routine

Validation uses the complete 2018–2019 period unless `QUICK_MODE=True`.

No optimizer update occurs during validation.

For the Shamal-aware model, the notebook also records regime classification accuracy as a training diagnostic, but checkpoint selection is still based on validation vector loss.


In [13]:
@torch.no_grad()
def validate_model(
    model,
):
    model.eval()

    total_sum = 0.0
    vector_sum = 0.0
    regime_sum = 0.0

    sample_count = 0

    regime_correct = 0
    regime_total = 0

    for batch in iter_split_batches(
        rows=VALIDATION_ROWS,
        batch_size=BATCH_HOURS,
        shuffle=False,
        epoch_seed=SEED,
    ):

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED
        ):
            output = model(
                history=batch["history"],
                static_node=STATIC_NODE_TENSOR,
                calendar=batch["calendar"],
            )

            losses = compute_losses(
                output=output,
                target_vector=batch["target"],
                target_regime=batch["regime_target"],
            )

        batch_size = (
            batch[
                "history"
            ].shape[0]
        )

        total_sum += (
            float(
                losses[
                    "total"
                ].detach().cpu()
            )
            * batch_size
        )

        vector_sum += (
            float(
                losses[
                    "vector"
                ].detach().cpu()
            )
            * batch_size
        )

        regime_sum += (
            float(
                losses[
                    "regime"
                ].detach().cpu()
            )
            * batch_size
        )

        sample_count += (
            batch_size
        )

        if (
            output[
                "regime_logits"
            ]
            is not None
        ):
            predicted_regime = (
                output[
                    "regime_logits"
                ]
                .argmax(
                    dim=-1
                )
            )

            regime_correct += int(
                (
                    predicted_regime
                    == batch[
                        "regime_target"
                    ]
                )
                .sum()
                .detach()
                .cpu()
            )

            regime_total += int(
                batch[
                    "regime_target"
                ].numel()
            )

        del (
            output,
            losses,
            batch
        )

    return {
        "total_loss": (
            total_sum
            / sample_count
        ),
        "vector_loss": (
            vector_sum
            / sample_count
        ),
        "regime_loss": (
            regime_sum
            / sample_count
        ),
        "regime_accuracy": (
            regime_correct
            / regime_total
            if regime_total > 0
            else np.nan
        ),
        "issue_times": sample_count,
    }


## 13. Full training function

Each model is trained independently.

Training features are shuffled within each annual file. Validation remains chronological.

For every model, the notebook saves:

- best checkpoint,
- training history CSV,
- best validation vector loss,
- best epoch,
- total runtime,
- parameter count.

Early stopping is based only on validation vector loss.


In [14]:
from tqdm.auto import tqdm


def train_one_model(
    model_name,
):

    print(
        "\n"
        + "=" * 90,
        flush=True
    )

    print(
        "TRAINING:",
        model_name,
        flush=True
    )

    print(
        "=" * 90,
        flush=True
    )


    # ---------------------------------------------------------
    # Build model
    # ---------------------------------------------------------

    model = build_model(
        model_name
    ).to(
        DEVICE
    )


    total_parameters = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


    print(
        f"Total parameters: "
        f"{total_parameters:,}"
    )

    print(
        f"Trainable parameters: "
        f"{trainable_parameters:,}"
    )

    print(
        f"Device: {DEVICE}"
    )

    if DEVICE.type == "cuda":

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    print(
        f"Batch issue-times: {BATCH_HOURS}"
    )

    print(
        f"Maximum epochs: {MAX_EPOCHS}"
    )

    print(
        f"Early stopping patience: "
        f"{EARLY_STOPPING_PATIENCE}"
    )

    print()


    # ---------------------------------------------------------
    # Optimizer
    # ---------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )


    # ---------------------------------------------------------
    # Learning-rate scheduler
    # ---------------------------------------------------------

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=LR_REDUCTION_FACTOR,
            patience=LR_REDUCTION_PATIENCE,
            min_lr=MIN_LEARNING_RATE,
        )
    )


    # ---------------------------------------------------------
    # Modern mixed-precision scaler
    # ---------------------------------------------------------

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP_ENABLED
    )


    # ---------------------------------------------------------
    # Training state
    # ---------------------------------------------------------

    best_validation_vector_loss = np.inf

    best_epoch = None

    epochs_without_improvement = 0

    history_rows = []

    training_start_time = time.time()


    # ---------------------------------------------------------
    # Output paths
    # ---------------------------------------------------------

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{model_name}.pt"
    )

    history_path = (
        TRAINING_TABLE_DIR
        / f"{model_name}_training_history.csv"
    )


    print(
        "Best checkpoint will be saved to:"
    )

    print(
        checkpoint_path
    )

    print()


    # =========================================================
    # Epoch loop
    # =========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        epoch_start_time = time.time()


        print(
            "\n"
            + "-" * 90
        )

        print(
            f"EPOCH {epoch}/{MAX_EPOCHS}"
        )

        print(
            "-" * 90
        )


        # =====================================================
        # TRAINING
        # =====================================================

        model.train()


        train_total_sum = 0.0

        train_vector_sum = 0.0

        train_regime_sum = 0.0

        train_issue_times = 0


        batch_iterator = iter_split_batches(
            rows=TRAIN_ROWS,
            batch_size=BATCH_HOURS,
            shuffle=True,
            epoch_seed=(
                SEED
                + epoch
            ),
        )


        progress_bar = tqdm(
            batch_iterator,
            desc=(
                f"{model_name} "
                f"E{epoch:02d} TRAIN"
            ),
            unit="batch",
            leave=True,
            dynamic_ncols=True,
        )


        for batch_number, batch in enumerate(
            progress_bar,
            start=1
        ):

            optimizer.zero_grad(
                set_to_none=True
            )


            # -------------------------------------------------
            # Forward pass
            # -------------------------------------------------

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=AMP_ENABLED
            ):

                output = model(
                    history=batch[
                        "history"
                    ],
                    static_node=STATIC_NODE_TENSOR,
                    calendar=batch[
                        "calendar"
                    ],
                )


                losses = compute_losses(
                    output=output,
                    target_vector=batch[
                        "target"
                    ],
                    target_regime=batch[
                        "regime_target"
                    ],
                )


            # -------------------------------------------------
            # Backward pass
            # -------------------------------------------------

            scaler.scale(
                losses[
                    "total"
                ]
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM,
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            # -------------------------------------------------
            # Batch statistics
            # -------------------------------------------------

            batch_size = (
                batch[
                    "history"
                ].shape[0]
            )


            batch_total_loss = float(
                losses[
                    "total"
                ]
                .detach()
                .cpu()
            )


            batch_vector_loss = float(
                losses[
                    "vector"
                ]
                .detach()
                .cpu()
            )


            batch_regime_loss = float(
                losses[
                    "regime"
                ]
                .detach()
                .cpu()
            )


            train_total_sum += (
                batch_total_loss
                * batch_size
            )


            train_vector_sum += (
                batch_vector_loss
                * batch_size
            )


            train_regime_sum += (
                batch_regime_loss
                * batch_size
            )


            train_issue_times += (
                batch_size
            )


            # -------------------------------------------------
            # Running averages
            # -------------------------------------------------

            running_total_loss = (
                train_total_sum
                / train_issue_times
            )


            running_vector_loss = (
                train_vector_sum
                / train_issue_times
            )


            running_regime_loss = (
                train_regime_sum
                / train_issue_times
            )


            # -------------------------------------------------
            # Live progress display
            # -------------------------------------------------

            progress_bar.set_postfix(
                {
                    "vec": (
                        f"{running_vector_loss:.4f}"
                    ),
                    "total": (
                        f"{running_total_loss:.4f}"
                    ),
                    "reg": (
                        f"{running_regime_loss:.4f}"
                    ),
                    "issues": (
                        f"{train_issue_times:,}"
                    ),
                },
                refresh=True
            )


            # -------------------------------------------------
            # Free batch memory
            # -------------------------------------------------

            del (
                output,
                losses,
                batch
            )


        progress_bar.close()


        # -----------------------------------------------------
        # Final training losses for epoch
        # -----------------------------------------------------

        train_total_loss = (
            train_total_sum
            / train_issue_times
        )


        train_vector_loss = (
            train_vector_sum
            / train_issue_times
        )


        train_regime_loss = (
            train_regime_sum
            / train_issue_times
        )


        print()

        print(
            "Training finished for this epoch."
        )

        print(
            f"Train total loss  : "
            f"{train_total_loss:.6f}"
        )

        print(
            f"Train vector loss : "
            f"{train_vector_loss:.6f}"
        )

        print(
            f"Train regime loss : "
            f"{train_regime_loss:.6f}"
        )


        # =====================================================
        # VALIDATION
        # =====================================================

        print(
            "\nRunning validation...",
            flush=True
        )


        validation_start = time.time()


        validation = validate_model(
            model
        )


        validation_seconds = (
            time.time()
            - validation_start
        )


        print(
            "Validation completed."
        )


        # -----------------------------------------------------
        # Scheduler
        # -----------------------------------------------------

        scheduler.step(
            validation[
                "vector_loss"
            ]
        )


        current_lr = (
            optimizer
            .param_groups[0][
                "lr"
            ]
        )


        # -----------------------------------------------------
        # Epoch timing
        # -----------------------------------------------------

        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


        epoch_seconds = (
            time.time()
            - epoch_start_time
        )


        # -----------------------------------------------------
        # Save history
        # -----------------------------------------------------

        history_rows.append(
            {
                "model": model_name,

                "epoch": epoch,

                "train_total_loss":
                    train_total_loss,

                "train_vector_loss":
                    train_vector_loss,

                "train_regime_loss":
                    train_regime_loss,

                "validation_total_loss":
                    validation[
                        "total_loss"
                    ],

                "validation_vector_loss":
                    validation[
                        "vector_loss"
                    ],

                "validation_regime_loss":
                    validation[
                        "regime_loss"
                    ],

                "validation_regime_accuracy":
                    validation[
                        "regime_accuracy"
                    ],

                "learning_rate":
                    current_lr,

                "epoch_seconds":
                    epoch_seconds,

                "validation_seconds":
                    validation_seconds,
            }
        )


        # -----------------------------------------------------
        # Epoch summary
        # -----------------------------------------------------

        print(
            "\nEPOCH SUMMARY"
        )

        print(
            f"Epoch                 : "
            f"{epoch}/{MAX_EPOCHS}"
        )

        print(
            f"Train vector loss     : "
            f"{train_vector_loss:.6f}"
        )

        print(
            f"Validation vector loss: "
            f"{validation['vector_loss']:.6f}"
        )

        print(
            f"Train total loss      : "
            f"{train_total_loss:.6f}"
        )

        print(
            f"Validation total loss : "
            f"{validation['total_loss']:.6f}"
        )

        print(
            f"Train regime loss     : "
            f"{train_regime_loss:.6f}"
        )

        print(
            f"Validation regime loss: "
            f"{validation['regime_loss']:.6f}"
        )

        print(
            f"Validation regime acc : "
            f"{validation['regime_accuracy']:.4f}"
        )

        print(
            f"Learning rate         : "
            f"{current_lr:.2e}"
        )

        print(
            f"Validation time       : "
            f"{validation_seconds / 60:.2f} min"
        )

        print(
            f"Epoch time            : "
            f"{epoch_seconds / 60:.2f} min"
        )


        # =====================================================
        # BEST MODEL CHECKPOINT
        # =====================================================

        if (
            validation[
                "vector_loss"
            ]
            < best_validation_vector_loss
        ):

            improvement = (
                best_validation_vector_loss
                - validation[
                    "vector_loss"
                ]
            )


            best_validation_vector_loss = (
                validation[
                    "vector_loss"
                ]
            )


            best_epoch = (
                epoch
            )


            epochs_without_improvement = 0


            torch.save(
                {
                    "model_name":
                        model_name,

                    "model_state_dict":
                        model.state_dict(),

                    "best_epoch":
                        best_epoch,

                    "best_validation_vector_loss":
                        best_validation_vector_loss,

                    "all_lags":
                        ALL_LAGS,

                    "forecast_horizons":
                        FORECAST_HORIZONS,

                    "scale_to_lags":
                        SCALE_TO_LAGS,

                    "d_model":
                        D_MODEL,

                    "n_heads":
                        N_HEADS,

                    "n_temporal_layers":
                        N_TEMPORAL_LAYERS,

                    "graph_layers":
                        GRAPH_LAYERS,

                    "dropout":
                        DROPOUT,

                    "k_neighbors":
                        K_NEIGHBORS,

                    "seed":
                        SEED,
                },
                checkpoint_path
            )


            print(
                "\nNEW BEST MODEL"
            )

            print(
                f"Best epoch: "
                f"{best_epoch}"
            )

            print(
                f"Best validation vector loss: "
                f"{best_validation_vector_loss:.6f}"
            )


            if np.isfinite(
                improvement
            ):

                print(
                    f"Improvement: "
                    f"{improvement:.6f}"
                )


            print(
                "Checkpoint saved to:"
            )

            print(
                checkpoint_path
            )


        else:

            epochs_without_improvement += 1


            print(
                "\nNo validation improvement."
            )

            print(
                "Early-stopping counter:",
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}"
            )

            print(
                f"Best validation vector loss: "
                f"{best_validation_vector_loss:.6f}"
            )

            print(
                f"Best epoch: "
                f"{best_epoch}"
            )


        # -----------------------------------------------------
        # Save history after every epoch to Google Drive
        # -----------------------------------------------------

        pd.DataFrame(
            history_rows
        ).to_csv(
            history_path,
            index=False
        )


        print(
            "\nTraining history saved to:"
        )

        print(
            history_path
        )


        # =====================================================
        # EARLY STOPPING
        # =====================================================

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                "\n"
                + "=" * 90
            )

            print(
                "EARLY STOPPING TRIGGERED"
            )

            print(
                f"Best epoch: "
                f"{best_epoch}"
            )

            print(
                f"Best validation vector loss: "
                f"{best_validation_vector_loss:.6f}"
            )

            print(
                "=" * 90
            )

            break


        # -----------------------------------------------------
        # Clean CUDA cache
        # -----------------------------------------------------

        if (
            DEVICE.type == "cuda"
        ):

            torch.cuda.empty_cache()


    # =========================================================
    # TRAINING COMPLETE
    # =========================================================

    total_training_seconds = (
        time.time()
        - training_start_time
    )


    print(
        "\n"
        + "=" * 90
    )

    print(
        f"TRAINING COMPLETE: {model_name}"
    )

    print(
        f"Best epoch: "
        f"{best_epoch}"
    )

    print(
        f"Best validation vector loss: "
        f"{best_validation_vector_loss:.6f}"
    )

    print(
        f"Total training time: "
        f"{total_training_seconds / 3600:.2f} hours"
    )

    print(
        "=" * 90
    )


    # ---------------------------------------------------------
    # Reload best checkpoint
    # ---------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    # ---------------------------------------------------------
    # Result summary
    # ---------------------------------------------------------

    result = {
        "model":
            model_name,

        "best_epoch":
            best_epoch,

        "best_validation_vector_loss":
            best_validation_vector_loss,

        "epochs_completed":
            len(
                history_rows
            ),

        "training_seconds":
            total_training_seconds,

        "training_hours":
            (
                total_training_seconds
                / 3600.0
            ),

        "checkpoint":
            str(
                checkpoint_path
            ),

        "history_file":
            str(
                history_path
            ),
    }


    # ---------------------------------------------------------
    # Free memory
    # ---------------------------------------------------------

    del (
        model,
        optimizer,
        scheduler,
        scaler,
        checkpoint
    )


    gc.collect()


    if (
        DEVICE.type == "cuda"
    ):

        torch.cuda.empty_cache()


    return result

## 14. Train all article models

This is the computationally expensive section.

For the final paper:

- keep `QUICK_MODE = False`,
- run every model to early stopping,
- do not interrupt a model and selectively retain a favorable partial run,
- retain every training-history CSV and checkpoint.

If GPU memory is insufficient, reduce `BATCH_HOURS`. Changing batch size is acceptable, but the final value must be reported in the manuscript and configuration output.


In [ ]:
training_results = []


for model_name in MODELS_TO_TRAIN:

    result = train_one_model(
        model_name
    )

    training_results.append(
        result
    )

    training_summary_live = pd.DataFrame(
        training_results
    )

    training_summary_live.to_csv(
        TRAINING_TABLE_DIR
        / "training_summary_partial.csv",
        index=False
    )


training_summary = pd.DataFrame(
    training_results
)


print(
    "\nAll requested training runs completed."
)

display(
    training_summary
)



TRAINING: GRU
Total parameters: 52,042
Trainable parameters: 52,042
Device: cuda
GPU: Tesla T4
Batch issue-times: 16
Maximum epochs: 20
Early stopping patience: 4

Best checkpoint will be saved to:
/content/drive/MyDrive/wind_model_results/weights/GRU.pt


------------------------------------------------------------------------------------------
EPOCH 1/20
------------------------------------------------------------------------------------------


GRU E01 TRAIN: 0batch [00:00, ?batch/s]


Training finished for this epoch.
Train total loss  : 0.207328
Train vector loss : 0.207328
Train regime loss : 0.000000

Running validation...


/tmp/ipykernel_6336/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_6336/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Validation completed.

EPOCH SUMMARY
Epoch                 : 1/20
Train vector loss     : 0.207328
Validation vector loss: 0.192318
Train total loss      : 0.207328
Validation total loss : 0.192318
Train regime loss     : 0.000000
Validation regime loss: 0.000000
Validation regime acc : nan
Learning rate         : 3.00e-04
Validation time       : 0.71 min
Epoch time            : 5.84 min

NEW BEST MODEL
Best epoch: 1
Best validation vector loss: 0.192318
Checkpoint saved to:
/content/drive/MyDrive/wind_model_results/weights/GRU.pt

Training history saved to:
/content/drive/MyDrive/wind_model_results/training_tables/GRU_training_history.csv

------------------------------------------------------------------------------------------
EPOCH 2/20
------------------------------------------------------------------------------------------


...

MS_NoGraph E02 TRAIN: 0batch [00:00, ?batch/s]


Training finished for this epoch.
Train total loss  : 0.186935
Train vector loss : 0.186935
Train regime loss : 0.000000

Running validation...


/tmp/ipykernel_6336/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_6336/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Validation completed.

EPOCH SUMMARY
Epoch                 : 2/20
Train vector loss     : 0.186935
Validation vector loss: 0.196628
Train total loss      : 0.186935
Validation total loss : 0.196628
Train regime loss     : 0.000000
Validation regime loss: 0.000000
Validation regime acc : nan
Learning rate         : 3.00e-04
Validation time       : 0.76 min
Epoch time            : 14.83 min

No validation improvement.
Early-stopping counter: 1/4
Best validation vector loss: 0.193261
Best epoch: 1

Training history saved to:
/content/drive/MyDrive/wind_model_results/training_tables/MS_NoGraph_training_history.csv

------------------------------------------------------------------------------------------
EPOCH 3/20
------------------------------------------------------------------------------------------


MS_NoGraph E03 TRAIN: 0batch [00:00, ?batch/s]

In [ ]:
training_results = []


for model_name in MODELS_TO_TRAIN:

    result = train_one_model(
        model_name
    )

    training_results.append(
        result
    )

    training_summary_live = pd.DataFrame(
        training_results
    )

    training_summary_live.to_csv(
        TRAINING_TABLE_DIR
        / "training_summary_partial.csv",
        index=False
    )


training_summary = pd.DataFrame(
    training_results
)


print(
    "\nAll requested training runs completed."
)

display(
    training_summary
)



TRAINING: MS_NoGraph
Total parameters: 350,915
Trainable parameters: 350,915
Device: cuda
GPU: Tesla T4
Batch issue-times: 16
Maximum epochs: 20
Early stopping patience: 4

Best checkpoint will be saved to:
/content/drive/MyDrive/wind_model_results/weights/MS_NoGraph.pt


------------------------------------------------------------------------------------------
EPOCH 1/20
------------------------------------------------------------------------------------------


/tmp/ipykernel_1721/824970097.py:125: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


MS_NoGraph E01 TRAIN: 0batch [00:00, ?batch/s]


Training finished for this epoch.
Train total loss  : 0.209086
Train vector loss : 0.209086
Train regime loss : 0.000000

Running validation...


/tmp/ipykernel_1721/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_1721/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


...

MS_NoGraph E08 TRAIN: 0batch [00:00, ?batch/s]


Training finished for this epoch.
Train total loss  : 0.172810
Train vector loss : 0.172810
Train regime loss : 0.000000

Running validation...


/tmp/ipykernel_1721/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_1721/2591984336.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Validation completed.

EPOCH SUMMARY
Epoch                 : 8/20
Train vector loss     : 0.172810
Validation vector loss: 0.188306
Train total loss      : 0.172810
Validation total loss : 0.188306
Train regime loss     : 0.000000
Validation regime loss: 0.000000
Validation regime acc : nan
Learning rate         : 3.00e-04
Validation time       : 0.73 min
Epoch time            : 14.55 min

No validation improvement.
Early-stopping counter: 1/4
Best validation vector loss: 0.183763
Best epoch: 7

Training history saved to:
/content/drive/MyDrive/wind_model_results/training_tables/MS_NoGraph_training_history.csv

------------------------------------------------------------------------------------------
EPOCH 9/20
------------------------------------------------------------------------------------------


MS_NoGraph E09 TRAIN: 0batch [00:00, ?batch/s]

In [ ]:
training_results = []


for model_name in MODELS_TO_TRAIN:

    result = train_one_model(
        model_name
    )

    training_results.append(
        result
    )

    training_summary_live = pd.DataFrame(
        training_results
    )

    training_summary_live.to_csv(
        TRAINING_TABLE_DIR
        / "training_summary_partial.csv",
        index=False
    )


training_summary = pd.DataFrame(
    training_results
)


print(
    "\nAll requested training runs completed."
)

display(
    training_summary
)


## 15. Save the complete experiment configuration

The experiment configuration is written to JSON and CSV so the manuscript can report the final architecture and training protocol exactly.

The independent test year is recorded as reserved but has not been evaluated.


In [ ]:
EXPERIMENT_CONFIG_JSON = (
    TRAINING_TABLE_DIR
    / "training_experiment_config.json"
)

EXPERIMENT_CONFIG_CSV = (
    TRAINING_TABLE_DIR
    / "training_experiment_config.csv"
)

TRAINING_SUMMARY_PATH = (
    TRAINING_TABLE_DIR
    / "training_summary.csv"
)


experiment_config = {
    "seed": SEED,
    "device_used": str(
        DEVICE
    ),
    "quick_mode": QUICK_MODE,
    "batch_issue_times": BATCH_HOURS,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRAD_CLIP_NORM,
    "huber_beta": HUBER_BETA,
    "regime_loss_weight": REGIME_LOSS_WEIGHT,
    "class_weight_power": CLASS_WEIGHT_POWER,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_temporal_layers": N_TEMPORAL_LAYERS,
    "graph_layers": GRAPH_LAYERS,
    "dropout": DROPOUT,
    "k_neighbors": K_NEIGHBORS,
    "lags_hours": ALL_LAGS,
    "forecast_horizons_hours": FORECAST_HORIZONS,
    "scale_to_lags": SCALE_TO_LAGS,
    "training_years": (
        TRAIN_ROWS[
            "year"
        ]
        .astype(int)
        .tolist()
    ),
    "validation_years": (
        VALIDATION_ROWS[
            "year"
        ]
        .astype(int)
        .tolist()
    ),
    "reserved_test_years": (
        TEST_ROWS[
            "year"
        ]
        .astype(int)
        .tolist()
    ),
    "model_names": MODEL_NAMES,
    "models_trained_this_run": MODELS_TO_TRAIN,
    "primary_checkpoint_metric": (
        "validation_vector_huber_loss"
    ),
    "true_future_shamal_label_used_as_predictor": False,
}


with open(
    EXPERIMENT_CONFIG_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=2
    )


flat_config_rows = []

for key, value in experiment_config.items():

    if isinstance(
        value,
        (
            list,
            dict
        )
    ):
        value = json.dumps(
            value
        )

    flat_config_rows.append({
        "parameter": key,
        "value": value,
    })


pd.DataFrame(
    flat_config_rows
).to_csv(
    EXPERIMENT_CONFIG_CSV,
    index=False
)


training_summary.to_csv(
    TRAINING_SUMMARY_PATH,
    index=False
)


print("Saved:")
print(EXPERIMENT_CONFIG_JSON.resolve())
print(EXPERIMENT_CONFIG_CSV.resolve())
print(TRAINING_SUMMARY_PATH.resolve())


## 16. Final training integrity checks

The final checks enforce the experimental boundary between model development and final testing.

This notebook must finish with:

- all six model checkpoints present,
- a training history for every model,
- no test-year training,
- no test-year validation,
- no true future Shamal labels used as direct predictors,
- `QUICK_MODE=False` for publication runs.

The next notebook will load the frozen checkpoints and evaluate them once on 2020.


In [ ]:
print(
    "Running final training integrity checks ..."
)


assert set(
    TRAIN_ROWS[
        "year"
    ].astype(int)
) == set(
    range(
        2005,
        2018
    )
)

assert set(
    VALIDATION_ROWS[
        "year"
    ].astype(int)
) == {
    2018,
    2019,
}

assert set(
    TEST_ROWS[
        "year"
    ].astype(int)
) == {
    2020,
}


assert not (
    TRAIN_ROWS[
        "year"
    ]
    == 2020
).any()

assert not (
    VALIDATION_ROWS[
        "year"
    ]
    == 2020
).any()


for model_name in MODELS_TO_TRAIN:

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{model_name}.pt"
    )

    history_path = (
        TRAINING_TABLE_DIR
        / f"{model_name}_training_history.csv"
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: {checkpoint_path}"
    )

    assert history_path.exists(), (
        f"Missing training history: {history_path}"
    )


assert (
    experiment_config[
        "true_future_shamal_label_used_as_predictor"
    ]
    is False
)


if QUICK_MODE:
    print(
        "\nWARNING: QUICK_MODE=True."
    )

    print(
        "This run is a pipeline/debug run and must not "
        "be used for manuscript results."
    )

else:
    print(
        "\nPublication-scale mode confirmed."
    )


print(
    "\nTraining/test separation: PASSED"
)

print(
    "Checkpoint integrity: PASSED"
)

print(
    "Shamal leakage rule: PASSED"
)

print(
    "\n06_full_model_training.ipynb: COMPLETED"
)

print(
    "\nDo not retrain models after inspecting the "
    "2020 results unless the test set is replaced by "
    "a new untouched final holdout."
)


## Save and download the trained weights

Kaggle inputs under `/kaggle/input/` are read-only. All outputs are saved under `/kaggle/working/`.

Best model checkpoints are stored at:

`/kaggle/working/wind_training_outputs/weights/`

The next cell:

1. lists every `.pt` checkpoint,
2. copies the proposed final model to `SAMGT_Full_best_weights.pt`,
3. zips all weights, histories, graph metadata, and experiment configuration,
4. displays download links in the notebook.

After training, also use Kaggle **Save Version / Save & Run All** so the output files are retained by Kaggle.


In [ ]:
import shutil
from IPython.display import FileLink, display

weight_files = sorted(
    CHECKPOINT_DIR.glob("*.pt")
)

print("Saved weight files:")
for path in weight_files:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f" - {path.name}: {size_mb:.2f} MB")

samgt_source = CHECKPOINT_DIR / "SAMGT_Full.pt"
samgt_download_copy = KAGGLE_WORKING_ROOT / "SAMGT_Full_best_weights.pt"

if samgt_source.exists():
    shutil.copy2(
        samgt_source,
        samgt_download_copy
    )
    print("\\nCreated:", samgt_download_copy)

archive_base = KAGGLE_WORKING_ROOT / "wind_training_outputs_bundle"

archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(OUTPUT_ROOT),
    )
)

print("\\nCreated complete training archive:")
print(archive_path)
print(
    f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB"
)

print("\\nDownload links:")

if samgt_download_copy.exists():
    display(FileLink(str(samgt_download_copy)))

display(FileLink(str(archive_path)))


## Notebook 06 complete — Kaggle version

When training finishes successfully, preserve these two convenient outputs:

- `/kaggle/working/SAMGT_Full_best_weights.pt` — proposed model checkpoint,
- `/kaggle/working/wind_training_outputs_bundle.zip` — all weights, histories, graph files, and configuration.

The complete working output is under:

`/kaggle/working/wind_training_outputs/`

For the article, keep the complete ZIP, not only the final SAMGT file. Notebook 07 needs the baseline and ablation checkpoints for the full comparison.

The next step is the frozen independent 2020 evaluation.
